In [7]:
SCENES_LIGHT = [
    "teapot_scene001",
    "grogu_scene002",
    "gnome_scene003",
    "car_scene004",
    "pitcher_scene005",
    "blocks_scene006",
    "cactus_scene007",
]

In [ ]:
import json
import re
import numpy as np
from pathlib import Path

SCORES_KEY_MAP = {
    "view_all": "view",
    "light_all": "light",
    "geometry_all": "geometry",
    "material_all": "material",
    "shape_all": "shape",
}

# Normalize both sides so mismatched names still match
NAME_ALIASES = {
    "car": "cart",
}

def normalize(tok):
    return NAME_ALIASES.get(tok, tok)

def parse_scenes_light(scenes):
    """Split each 'name_sceneXXX' entry into (name_tokens, scene_token)."""
    parsed = []
    for s in scenes:
        parts = s.split("_")
        scene_tok = next(p for p in parts if re.fullmatch(r"scene\d+", p))
        name_toks = frozenset(
            normalize(p) for p in parts if not re.fullmatch(r"scene\d+", p)
        )
        parsed.append((name_toks, scene_tok))
    return parsed

def find_canonical(key: str, parsed_scenes) -> str | None:
    """Return the SCENES_LIGHT name that key matches, or None."""
    key_toks = set(normalize(t) for t in key.split("_"))
    for (name_toks, scene_tok), canonical in zip(parsed_scenes, SCENES_LIGHT):
        if scene_tok in key_toks and name_toks <= key_toks:
            return canonical
    return None

def filter_and_recompute(src_path: Path, dst_path: Path):
    with open(src_path) as f:
        data = json.load(f)

    parsed = parse_scenes_light(SCENES_LIGHT)

    filtered_scores = {}
    for scores_key in SCORES_KEY_MAP:
        subset = {}
        for scene, metrics in data["scores"][scores_key].items():
            canonical = find_canonical(scene, parsed)
            if canonical is not None:
                subset[canonical] = metrics
        filtered_scores[scores_key] = subset

    new_stats = {}
    for scores_key, stats_key in SCORES_KEY_MAP.items():
        subset = filtered_scores[scores_key]
        if not subset:
            new_stats[stats_key] = {"scene_count": 0}
            continue
        metric_names = list(next(iter(subset.values())).keys())
        entry = {"scene_count": len(subset)}
        for metric in metric_names:
            vals = [v[metric] for v in subset.values()]
            entry[metric] = [float(np.mean(vals)), float(np.std(vals))]
        new_stats[stats_key] = entry

    out = {"scores_stats": new_stats, "scores": filtered_scores}
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    with open(dst_path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Written: {dst_path}  ({sum(len(v) for v in filtered_scores.values())} entries across all categories)")

# eval_dir = Path(".")
# for src in eval_dir.glob("*.json"):
#     dst = src.with_stem(src.stem + "_subset")
#     filter_and_recompute(src, dst)


In [20]:
eval_dir = Path(".")
for src in eval_dir.glob("*.json"):
    dst = src.with_stem(src.stem + "_subset")
    filter_and_recompute(src, Path("subset") / dst)

{'view_all': {'blocks_scene006': {'psnr_hdr': 34.120735320543055, 'psnr_ldr': 38.729267689503835, 'lpips': 0.01849214071407914, 'ssim': 0.9896146595478058}, 'cactus_scene007': {'psnr_hdr': 28.664070322470117, 'psnr_ldr': 39.37394613364335, 'lpips': 0.01418729880824685, 'ssim': 0.9924874350894243}, 'car_scene004': {'psnr_hdr': 34.12541094768521, 'psnr_ldr': 38.482040746274585, 'lpips': 0.009825728880241514, 'ssim': 0.9899445627816021}, 'gnome_scene003': {'psnr_hdr': 35.0100705418891, 'psnr_ldr': 39.432218968827314, 'lpips': 0.048412651382386686, 'ssim': 0.9814251333475112}, 'grogu_scene002': {'psnr_hdr': 28.30278172639309, 'psnr_ldr': 35.83212647486492, 'lpips': 0.011846229247748852, 'ssim': 0.9935838916804641}, 'pitcher_scene005': {'psnr_hdr': 25.54275620798931, 'psnr_ldr': 32.90528162929616, 'lpips': 0.030714655108749867, 'ssim': 0.9778959622606636}, 'teapot_scene001': {'psnr_hdr': 28.418114483829335, 'psnr_ldr': 38.215278594647764, 'lpips': 0.007026630779728293, 'ssim': 0.99291752022